# 🔍 Notebook 03 — Exploratory Data Analysis (EDA)
## Sales Data Analytics | Superstore Sales Dataset

---

### 🎯 Objective
Perform comprehensive EDA to extract business insights:
1. Overall KPIs
2. Sales & Profit by Category, Sub-Category
3. Regional & State-wise Analysis
4. Customer Segment Analysis
5. Time Series Trends
6. Product Performance
7. Discount Impact Analysis
8. Shipping Analysis

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)

# Style configuration
plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'DejaVu Sans',
})

PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
VIZ_DIR = Path('../visualizations')
VIZ_DIR.mkdir(exist_ok=True)

# Load clean data
df = pd.read_csv('../dataset/superstore_clean.csv', parse_dates=['order_date', 'ship_date'])
print(f'✅ Clean dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 3.1 — Business KPIs Overview

In [ ]:
kpis = {
    'Total Orders'     : df['order_id'].nunique(),
    'Total Customers'  : df['customer_id'].nunique(),
    'Total Products'   : df['product_id'].nunique(),
    'Total Revenue'    : df['sales'].sum(),
    'Total Profit'     : df['profit'].sum(),
    'Profit Margin %'  : df['profit'].sum() / df['sales'].sum() * 100,
    'Avg Order Value'  : df.groupby('order_id')['sales'].sum().mean(),
    'Total Units Sold' : df['quantity'].sum(),
    'Avg Ship Days'    : df['shipping_days'].mean(),
}

print('=' * 55)
print('           SALES PERFORMANCE KPIs')
print('=' * 55)
for k, v in kpis.items():
    if 'Revenue' in k or 'Profit' in k or 'Value' in k:
        print(f'  {k:<22} : ${v:>12,.2f}')
    elif '%' in k:
        print(f'  {k:<22} : {v:>11.2f}%')
    elif 'Days' in k:
        print(f'  {k:<22} : {v:>11.1f} days')
    else:
        print(f'  {k:<22} : {int(v):>12,}')
print('=' * 55)

## 3.2 — Sales by Category

In [ ]:
cat_summary = (
    df.groupby('category')
    .agg(
        Total_Sales   = ('sales',     'sum'),
        Total_Profit  = ('profit',    'sum'),
        Total_Orders  = ('order_id',  'nunique'),
        Total_Units   = ('quantity',  'sum'),
    )
    .assign(Profit_Margin=lambda x: (x['Total_Profit'] / x['Total_Sales'] * 100).round(2))
    .sort_values('Total_Sales', ascending=False)
    .round(2)
)
print(cat_summary.to_string())

## 3.3 — Sub-Category Performance

In [ ]:
subcat = (
    df.groupby(['category', 'sub_category'])
    .agg(
        Total_Sales   = ('sales',    'sum'),
        Total_Profit  = ('profit',   'sum'),
        Total_Orders  = ('order_id', 'nunique'),
    )
    .assign(Profit_Margin=lambda x: (x['Total_Profit'] / x['Total_Sales'] * 100).round(2))
    .sort_values('Total_Sales', ascending=False)
    .round(2)
)
print('── Top 10 Sub-Categories by Sales ──')
print(subcat.head(10).to_string())
print('\n── Loss-Making Sub-Categories ──')
print(subcat[subcat['Total_Profit'] < 0].to_string())

## 3.4 — Regional Sales Analysis

In [ ]:
region_summary = (
    df.groupby('region')
    .agg(
        Total_Sales     = ('sales',       'sum'),
        Total_Profit    = ('profit',      'sum'),
        Total_Orders    = ('order_id',    'nunique'),
        Total_Customers = ('customer_id', 'nunique'),
        Avg_Order_Value = ('sales',       'mean'),
    )
    .assign(Profit_Margin=lambda x: (x['Total_Profit'] / x['Total_Sales'] * 100).round(2))
    .assign(Sales_Share=lambda x: (x['Total_Sales'] / x['Total_Sales'].sum() * 100).round(2))
    .sort_values('Total_Sales', ascending=False)
    .round(2)
)
print(region_summary.to_string())

## 3.5 — Top 10 States by Sales

In [ ]:
state_summary = (
    df.groupby(['state', 'region'])
    .agg(
        Total_Sales  = ('sales',    'sum'),
        Total_Profit = ('profit',   'sum'),
        Total_Orders = ('order_id', 'nunique'),
    )
    .assign(Profit_Margin=lambda x: (x['Total_Profit'] / x['Total_Sales'] * 100).round(2))
    .sort_values('Total_Sales', ascending=False)
    .round(2)
)
print('── Top 10 States by Revenue ──')
print(state_summary.head(10).to_string())
print('\n── Bottom 5 States by Revenue ──')
print(state_summary.tail(5).to_string())

## 3.6 — Customer Segment Analysis

In [ ]:
segment_summary = (
    df.groupby('segment')
    .agg(
        Total_Sales     = ('sales',       'sum'),
        Total_Profit    = ('profit',      'sum'),
        Total_Customers = ('customer_id', 'nunique'),
        Total_Orders    = ('order_id',    'nunique'),
        Avg_Order_Value = ('sales',       'mean'),
    )
    .assign(Profit_Margin=lambda x: (x['Total_Profit'] / x['Total_Sales'] * 100).round(2))
    .sort_values('Total_Sales', ascending=False)
    .round(2)
)
print(segment_summary.to_string())

## 3.7 — Monthly & Yearly Sales Trends

In [ ]:
# Yearly trend
yearly = (
    df.groupby('order_year')
    .agg(Total_Sales=('sales', 'sum'), Total_Profit=('profit', 'sum'), Total_Orders=('order_id', 'nunique'))
    .round(2)
)
yearly['YoY_Growth_%'] = yearly['Total_Sales'].pct_change() * 100
print('── Yearly Trend ──')
print(yearly.round(2).to_string())

# Monthly trend
monthly = (
    df.groupby(['order_year', 'order_month', 'month_name'])
    .agg(Total_Sales=('sales', 'sum'), Total_Profit=('profit', 'sum'))
    .reset_index()
    .sort_values(['order_year', 'order_month'])
    .round(2)
)
print('\n── Monthly Trend Sample ──')
print(monthly.tail(12).to_string(index=False))

## 3.8 — Top & Bottom Products

In [ ]:
product_perf = (
    df.groupby('product_name')
    .agg(
        Total_Sales   = ('sales',    'sum'),
        Total_Profit  = ('profit',   'sum'),
        Total_Units   = ('quantity', 'sum'),
        Order_Count   = ('order_id', 'nunique'),
    )
    .assign(Margin=lambda x: (x['Total_Profit'] / x['Total_Sales'] * 100).round(2))
    .sort_values('Total_Sales', ascending=False)
    .round(2)
)

print('── Top 10 Best-Selling Products ──')
print(product_perf.head(10)[['Total_Sales', 'Total_Profit', 'Margin', 'Total_Units']].to_string())

print('\n── Top 10 Most Profitable Products ──')
print(product_perf.sort_values('Total_Profit', ascending=False).head(10)[['Total_Sales', 'Total_Profit', 'Margin']].to_string())

print('\n── Top 10 Loss-Making Products ──')
print(product_perf.sort_values('Total_Profit', ascending=True).head(10)[['Total_Sales', 'Total_Profit', 'Margin']].to_string())

## 3.9 — Discount Impact Analysis

In [ ]:
# Correlation between discount and profit
corr_discount_profit = df['discount'].corr(df['profit'])
corr_discount_sales  = df['discount'].corr(df['sales'])
print(f'Correlation — Discount vs Profit : {corr_discount_profit:.4f}')
print(f'Correlation — Discount vs Sales  : {corr_discount_sales:.4f}')

# Discount band analysis
discount_analysis = (
    df.groupby('discount_band', observed=True)
    .agg(
        Count       = ('sales',   'count'),
        Avg_Sales   = ('sales',   'mean'),
        Avg_Profit  = ('profit',  'mean'),
        Total_Sales = ('sales',   'sum'),
        Total_Profit= ('profit',  'sum'),
        Loss_Count  = ('profit',  lambda x: (x < 0).sum()),
    )
    .assign(Profit_Margin=lambda x: (x['Total_Profit'] / x['Total_Sales'] * 100).round(2))
    .round(2)
)
print('\n── Discount Band Impact on Profit ──')
print(discount_analysis.to_string())

## 3.10 — Shipping Analysis

In [ ]:
ship_analysis = (
    df.groupby('ship_mode')
    .agg(
        Order_Count   = ('order_id',      'nunique'),
        Avg_Ship_Days = ('shipping_days', 'mean'),
        Min_Days      = ('shipping_days', 'min'),
        Max_Days      = ('shipping_days', 'max'),
        Avg_Sales     = ('sales',         'mean'),
        Total_Sales   = ('sales',         'sum'),
    )
    .sort_values('Avg_Ship_Days')
    .round(2)
)
print('── Shipping Mode Analysis ──')
print(ship_analysis.to_string())

# Orders with shipping > 5 days
late_shipments = df[df['shipping_days'] > 5]
print(f'\nOrders with shipping > 5 days: {late_shipments["order_id"].nunique():,} ({late_shipments["order_id"].nunique()/df["order_id"].nunique()*100:.1f}% of all orders)')

## 3.11 — Key EDA Findings

| # | Finding |
|---|---|
| 1 | **Technology** generates the highest revenue but **Office Supplies** has the highest order volume |
| 2 | **Furniture (Tables)** is the most loss-making sub-category — driven by heavy discounts |
| 3 | **West region** leads in sales; **Central** has the lowest profit margin |
| 4 | **Consumer segment** accounts for ~50% of revenue; **Corporate** has the highest avg order value |
| 5 | **Q4 (Oct–Dec)** is consistently the best-performing quarter — holiday seasonality |
| 6 | Discounts above **30% always result in negative profit** |
| 7 | **Same Day** shipping is the fastest but least used mode |
| 8 | **California** and **New York** are the top two states by revenue |

**➡️ Next Step**: Notebook 04 — Data Visualization